<a href="https://colab.research.google.com/github/kartik815/Amazon-ML-Challenge-2026/blob/main/notebooks/04_Blocking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive

drive.mount('/content/drive')

import os
import gc
import pandas as pd

DRIVE_ROOT = "/content/drive/MyDrive/Amazon ML Challenge 2026"

CLEANED_DATA_ROOT = os.path.join(
    DRIVE_ROOT,
    "03_Experiments",
    "Cleaned_Data"
)

TRAIN_ROOT = os.path.join(
    DRIVE_ROOT,
    "01_Dataset",
    "6ab10eb3b23ba_student_resource",
    "student_resource",
    "dataset",
    "train"
)

TEST_ROOT = os.path.join(
    DRIVE_ROOT,
    "01_Dataset",
    "6ab10eb3b23ba_student_resource",
    "student_resource",
    "dataset",
    "test"
)

BLOCKING_OUTPUT_ROOT = os.path.join(
    DRIVE_ROOT,
    "05_Outputs",
    "Candidate_Pairs"
)

os.makedirs(BLOCKING_OUTPUT_ROOT, exist_ok=True)

print("Cleaned data:")
print(CLEANED_DATA_ROOT)

print("\nBlocking output:")
print(BLOCKING_OUTPUT_ROOT)

Mounted at /content/drive
Cleaned data:
/content/drive/MyDrive/Amazon ML Challenge 2026/03_Experiments/Cleaned_Data

Blocking output:
/content/drive/MyDrive/Amazon ML Challenge 2026/05_Outputs/Candidate_Pairs


In [3]:
CLEANED_FILES = [
    "train_s1_cleaned.tsv",
    "train_s2_cleaned.tsv",
    "train_s3_cleaned.tsv",
    "test_s1_cleaned.tsv",
    "test_s2_cleaned.tsv",
    "test_s3_cleaned.tsv"
]

print("Checking cleaned files...\n")

for filename in CLEANED_FILES:

    path = os.path.join(
        CLEANED_DATA_ROOT,
        filename
    )

    print(
        f"{filename:<25}",
        "EXISTS" if os.path.exists(path) else "MISSING"
    )

Checking cleaned files...

train_s1_cleaned.tsv      EXISTS
train_s2_cleaned.tsv      EXISTS
train_s3_cleaned.tsv      EXISTS
test_s1_cleaned.tsv       EXISTS
test_s2_cleaned.tsv       EXISTS
test_s3_cleaned.tsv       EXISTS


In [4]:
sample_path = os.path.join(
    CLEANED_DATA_ROOT,
    "train_s1_cleaned.tsv"
)

sample = pd.read_csv(
    sample_path,
    sep="\t",
    nrows=5
)

print("Columns:")
print(sample.columns.tolist())

print("\nSample:")
display(sample)

Columns:
['entity_id', 'business_name', 'business_address', 'country', 'name_norm', 'address_norm', 'country_norm', 'name_missing', 'address_missing', 'name_token_count', 'address_token_count', 'name_core']

Sample:


,entity_id,business_name,business_address,country,name_norm,address_norm,country_norm,name_missing,address_missing,name_token_count,address_token_count,name_core
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US,orelee s barbershop,1795 westchester dr high point nc,us,0,0,3,6,orelee s barbershop
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US,prime money,17560 ellis rd tahlequah ok,us,0,0,2,5,prime money
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US,b retail inc,1712 montebello ave phoenix az,us,0,0,3,5,b retail
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US,christ chapel,2100 cameron dr unit apt g dundalk md,us,0,0,2,8,christ chapel
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India,prabhav business center,797 lake town block a kolkata howrah west bengal,india,0,0,3,9,prabhav business center


Blocker 1 (country_norm + name_core)

In [5]:
import pandas as pd
from collections import Counter

S2_PATH = os.path.join(
    CLEANED_DATA_ROOT,
    "train_s2_cleaned.tsv"
)

S3_PATH = os.path.join(
    CLEANED_DATA_ROOT,
    "train_s3_cleaned.tsv"
)

CHUNK_SIZE = 200_000

def get_block_sizes(path):
  counter = Counter()

  for chunk in pd.read_csv(
      path,
      sep="\t",
      chunksize=CHUNK_SIZE,
      dtype={"country_norm": "string", "name_norm": "string"}):
      chunk = chunk[
          chunk["name_core"].notna() &
          (chunk["name_core"] != "")
      ]

      keys = zip(
          chunk["country_norm"],
          chunk["name_core"]
      )

      counter.update(keys)

      del chunk
      gc.collect()

  return counter

print("Building S2 block statistics...")
s2_blocks = get_block_sizes(S2_PATH)

print("Building S3 block statistics...")
s3_blocks = get_block_sizes(S3_PATH)

print("\nS2 unique blocks:", len(s2_blocks))
print("S3 unique blocks:", len(s3_blocks))


Building S2 block statistics...
Building S3 block statistics...

S2 unique blocks: 3597772
S3 unique blocks: 3850257


In [6]:
print("\nLargest S2 blocks:")
for key, count in s2_blocks.most_common(20):
    print(count, "->", key)

print("\nLargest S3 blocks:")
for key, count in s3_blocks.most_common(20):
    print(count, "->", key)


Largest S2 blocks:
647 -> ('us', 'meridian')
563 -> ('us', 'physical therapy')
550 -> ('us', 'primary care')
514 -> ('us', 'womens health')
507 -> ('us', 'internal medicine')
500 -> ('us', 'behavioral health')
494 -> ('us', 'urgent care')
491 -> ('us', 'pediatric dental')
482 -> ('us', 'pediatric dentistry')
408 -> ('us', 'ear nose throat')
398 -> ('us', 'foot ankle')
375 -> ('us', 'family center')
374 -> ('us', 'summit')
374 -> ('us', 'lynx')
366 -> ('us', 'anchor')
362 -> ('us', 'sapphire')
360 -> ('us', 'helios')
359 -> ('us', 'falcon')
356 -> ('us', 'cedar')
355 -> ('us', 'earnosethroat com')

Largest S3 blocks:
584 -> ('us', 'meridian')
548 -> ('us', 'primary care')
538 -> ('us', 'physical therapy')
524 -> ('us', 'pediatric dental')
507 -> ('us', 'urgent care')
492 -> ('us', 'womens health')
490 -> ('us', 'pediatric dentistry')
458 -> ('us', 'behavioral health')
452 -> ('us', 'internal medicine')
426 -> ('us', 'summit')
418 -> ('us', 'cascade')
412 -> ('us', 'family center')
399 

In [7]:
from collections import defaultdict
import gc

def build_block_index(path):

    index = defaultdict(list)

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["entity_id", "country_norm", "name_core"],
        chunksize=200_000,
        dtype={
            "entity_id": "string",
            "country_norm": "string",
            "name_core": "string"
        }
    ):

        chunk = chunk[
            chunk["name_core"].notna() &
            (chunk["name_core"] != "")
        ]

        for country, name, entity_id in zip(
            chunk["country_norm"],
            chunk["name_core"],
            chunk["entity_id"]
        ):
            index[(country, name)].append(entity_id)

        del chunk
        gc.collect()

    return index

In [8]:
print("Building S2 index...")
s2_index = build_block_index(S2_PATH)
print("S2 index blocks:", len(s2_index))
gc.collect()
print("\nBuilding S3 index...")
s3_index = build_block_index(S3_PATH)
print("S3 index blocks:", len(s3_index))

Building S2 index...
S2 index blocks: 3597772

Building S3 index...
S3 index blocks: 3850257


In [9]:
GT_PATH = os.path.join(
    TRAIN_ROOT,
    "train_ground_truth.tsv"
)

S1_PATH = os.path.join(
    CLEANED_DATA_ROOT,
    "train_s1_cleaned.tsv"
)

gt = pd.read_csv(
    GT_PATH,
    sep="\t",
    dtype={
        "source1_entity_id": "string",
        "matched_entity_ids": "string"
    }
)

s1 = pd.read_csv(
    S1_PATH,
    sep="\t",
    usecols=[
        "entity_id",
        "country_norm",
        "name_core"
    ],
    dtype={
        "entity_id": "string",
        "country_norm": "string",
        "name_core": "string"
    }
)

print("S1 rows:", len(s1))
print("Ground truth rows:", len(gt))

S1 rows: 2206821
Ground truth rows: 2206821


In [10]:
gt_lookup = {}

for row in gt.itertuples(index=False):
    s1_id = row.source1_entity_id
    matched = row.matched_entity_ids

    if pd.isna(matched) or matched == "":
        gt_lookup[s1_id] = set()
    else:
        gt_lookup[s1_id] = {
            x.strip()
            for x in str(matched).split(",")
            if x.strip()
        }

print("Ground truth lookup entries:", len(gt_lookup))

Ground truth lookup entries: 2206821


In [11]:
def calculate_block_recall_fast(s1_df, gt_lookup, index, prefix):

    total_matches = 0
    captured_matches = 0

    for row in s1_df.itertuples(index=False):

        s1_id = row.entity_id
        country = row.country_norm
        name = row.name_core

        true_ids = {
            x for x in gt_lookup.get(s1_id, set())
            if x.startswith(prefix)
        }

        if not true_ids:
            continue

        total_matches += len(true_ids)
        candidates = index.get((country, name), [])
        candidate_set = set(candidates)
        captured_matches += len(
            true_ids.intersection(candidate_set)
        )

    recall = (
        captured_matches / total_matches
        if total_matches
        else 0
    )

    return total_matches, captured_matches, recall

In [12]:
print("Evaluating S2 blocking recall...")

s2_total, s2_captured, s2_recall = calculate_block_recall_fast(
    s1,
    gt_lookup,
    s2_index,
    "S2-"
)

print("\nS2")
print("Total true matches :", s2_total)
print("Captured matches   :", s2_captured)
print(f"Blocking recall    : {s2_recall:.4%}")

Evaluating S2 blocking recall...

S2
Total true matches : 3693619
Captured matches   : 1447011
Blocking recall    : 39.1760%


In [13]:
print("Evaluating S3 blocking recall...")

s3_total, s3_captured, s3_recall = calculate_block_recall_fast(
    s1,
    gt_lookup,
    s3_index,
    "S3-"
)

print("\nS3")
print("Total true matches :", s3_total)
print("Captured matches   :", s3_captured)
print(f"Blocking recall    : {s3_recall:.4%}")

Evaluating S3 blocking recall...

S3
Total true matches : 3944746
Captured matches   : 1490930
Blocking recall    : 37.7953%


In [14]:
del s2_index
del s3_index
gc.collect()
print("Name indexes cleared from memory.")

Name indexes cleared from memory.


In [15]:
from collections import defaultdict
import gc
import pandas as pd

def build_address_index(path):
    index = defaultdict(list)

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["entity_id", "country_norm", "address_norm"],
        chunksize=200_000,
        dtype={
            "entity_id": "string",
            "country_norm": "string",
            "address_norm": "string"
        }
    ):
        chunk = chunk[
            chunk["address_norm"].notna() &
            (chunk["address_norm"] != "")
        ]

        for country, address, entity_id in zip(
            chunk["country_norm"],
            chunk["address_norm"],
            chunk["entity_id"]
        ):
            index[(country, address)].append(entity_id)

        del chunk
        gc.collect()

    return index

In [16]:
print("Building S2 address index...")
s2_address_index = build_address_index(S2_PATH)
print("S2 address blocks:", len(s2_address_index))

Building S2 address index...
S2 address blocks: 4090600


In [17]:
s2_addr_total, s2_addr_captured, s2_addr_recall = calculate_block_recall_fast(s1,gt_lookup,s2_address_index, "S2-")

print("S2 total true matches:", s2_addr_total)
print("S2 captured matches:", s2_addr_captured)
print(f"S2 address blocking recall: {s2_addr_recall:.4%}")

S2 total true matches: 3693619
S2 captured matches: 0
S2 address blocking recall: 0.0000%


In [18]:
del s2_address_index
gc.collect()
print("S2 address index cleared.")

S2 address index cleared.


In [19]:
print("Building S3 address index...")
s3_address_index = build_address_index(S3_PATH)
print("S3 address blocks:", len(s3_address_index))

Building S3 address index...
S3 address blocks: 4437787


In [20]:
s3_addr_total, s3_addr_captured, s3_addr_recall = calculate_block_recall_fast(s1, gt_lookup, s3_address_index, "S3-")

print("S3 total true matches:", s3_addr_total)
print("S3 captured matches:", s3_addr_captured)
print(f"S3 address blocking recall: {s3_addr_recall:.4%}")

S3 total true matches: 3944746
S3 captured matches: 0
S3 address blocking recall: 0.0000%


In [21]:
del s3_address_index
gc.collect()

150

In [22]:
# Show some S1 entities that have known S2 matches
# and compare their normalized name/address fields.

sample_s1_ids = []

for s1_id, matches in gt_lookup.items():
    s2_matches = [x for x in matches if x.startswith("S2-")]
    if s2_matches:
        sample_s1_ids.append((s1_id, s2_matches[:3]))

    if len(sample_s1_ids) >= 10:
        break

sample_s1_ids

[('S1-965667', ['S2-681193310', 'S2-743505751']),
 ('S1-55344266', ['S2-249013014', 'S2-197070651']),
 ('S1-343815751', ['S2-790675320', 'S2-479876582']),
 ('S1-656753428', ['S2-24659151', 'S2-153058913']),
 ('S1-102811957', ['S2-553508714', 'S2-478959098', 'S2-625774905']),
 ('S1-18727616', ['S2-755677256']),
 ('S1-318373630', ['S2-660036492']),
 ('S1-29845983', ['S2-648035184']),
 ('S1-789009573', ['S2-383871912']),
 ('S1-730934468', ['S2-356983532'])]

In [23]:
sample_ids = [x[0] for x in sample_s1_ids]
sample_s2_ids = [
    match
    for _, matches in sample_s1_ids
    for match in matches
]

s1_sample = pd.read_csv(
    S1_PATH,
    sep="\t",
    dtype="string"
)

s2_sample = pd.read_csv(
    S2_PATH,
    sep="\t",
    dtype="string"
)

s1_sample = s1_sample[
    s1_sample["entity_id"].isin(sample_ids)
]

s2_sample = s2_sample[
    s2_sample["entity_id"].isin(sample_s2_ids)
]

print("S1 samples:")
display(
    s1_sample[
        ["entity_id", "business_name", "business_address",
         "name_core", "address_norm"]
    ]
)

print("\nS2 matching samples:")
display(
    s2_sample[
        ["entity_id", "business_name", "business_address",
         "name_core", "address_norm"]
    ]
)

S1 samples:


,entity_id,business_name,business_address,name_core,address_norm
539414,S1-102811957,Payne Enterprises,"3315 Fremont Street, Peoria, IL",payne enterprises,3315 fremont st peoria il
547338,S1-343815751,Dahlia Power Reliable Scientific LLC,"630 45th Terrace, Kansas City, MO",dahlia power reliable scientific,630 45th terrace kansas city mo
603899,S1-318373630,Red Ventures Private Limited,"Rajasthan, Jaipur, Banipark, Gokul Apartment, ...",red ventures,rajasthan jaipur banipark gokul apt e 3a kanti...
1043869,S1-730934468,Orellana Investments LLC,"728 A Quail Avenue, Fl Ground Floor, Geneva, IA",orellana investments,728 a quail ave fl ground floor geneva ia
1072115,S1-656753428,Ss Food Private Limited,"Af-684, Nandgram Near Mother India Public Scho...",ss food,af 684 nandgram near mother india public schoo...
1232493,S1-29845983,Hendricks and Flowers Inc,"33 Sleepy Hollow Drive, Danbury, CT",hendricks and flowers,33 sleepy hollow dr danbury ct
1265942,S1-789009573,Hotel Enterprises Limited,"Wz-187C Shop No.13, 14 Kh. No.47 S/F. Vikaspur...",hotel enterprises,wz 187c shop no 13 14 kh no 47 s f vikaspuri b...
1286323,S1-965667,Maure Williams Colombier Inc,"85 Wayne Avenue, Ticonderoga, NY",maure williams colombier,85 wayne ave ticonderoga ny
1898165,S1-55344266,Raj Investments LLP,"6(29), C.I.T. Colony, 2Nd Main Road Mylapore, ...",raj investments,6 29 c i t colony 2nd main rd mylapore chennai...
2117972,S1-18727616,Lumay Boral,"1056 Belden Avenue, Akron, OH",lumay boral,1056 belden ave akron oh



S2 matching samples:


,entity_id,business_name,business_address,name_core,address_norm
191919,S2-648035184,Hendricks and Flowers Inc,"CT, SLEEPY HOLLOW DRIVE, DANBURY",hendricks and flowers,ct sleepy hollow dr danbury
223808,S2-755677256,Lumay Boral Inc.,"1056-1060 BELDEN AVE, PO BOX 8807, AKRON, OH",lumay boral,1056 1060 belden ave po box 8807 akron oh
385243,S2-625774905,PAYNE-ENRTPRMISES,"3315 FREMONT SAINT, PEORIA, IL",payne enrtprmises,3315 fremont saint peoria il
488130,S2-383871912,होटल एंटरप्राइजेज लिमिटेड,"WZ-187C SHOP NO.13, DELHI, WEST DELHI, Delhi",ह टल ए टरप र इज ज ल म ट ड,wz 187c shop no 13 delhi west delhi delhi
731099,S2-478959098,Payne Énterprises,"3315 FREMONT ST, PEORIA, IL",payne énterprises,3315 fremont st peoria il
809231,S2-660036492,रेड वेंचर्स प्राइवेट लिमिटेड,"G-1, BANIPARK, JAIPUR, Rajasthan",र ड व चर स प र इव ट ल म ट ड,g 1 banipark jaipur rajasthan
1376739,S2-681193310,Maure Wilblims Colombier Inc,<NA>,maure wilblims colombier,<NA>
1776076,S2-553508714,Payne Enterpires,"3315 FREMONT ST, PEORIA, IL",payne enterpires,3315 fremont st peoria il
1886844,S2-197070651,Raj Investments LLP,"6(29), C.I.T. COLONY, 2ND MAIN ROAD MYLAPORE, ...",raj investments,6 29 c i t colony 2nd main rd mylapore chennai...
1901972,S2-479876582,Dahlia Power Reliable Scientific,"45ND TERRACE, null, KANSAS CITY, MO",dahlia power reliable scientific,45nd terrace null kansas city mo


In [24]:
from collections import Counter
import pandas as pd
import gc

token_counts = Counter()

for chunk in pd.read_csv(
    S2_PATH,
    sep="\t",
    usecols=["name_core"],
    chunksize=200_000,
    dtype={"name_core": "string"}
):
    for name in chunk["name_core"].dropna():
        tokens = set(name.split())

        for token in tokens:
            if token:
                token_counts[token] += 1

    del chunk
    gc.collect()

print("Unique name tokens:", len(token_counts))

print("\nMost common tokens:")
for token, count in token_counts.most_common(30):
    print(f"{token:30} {count:,}")

Unique name tokens: 810506

Most common tokens:
ल                              237,102
ट                              232,862
र                              226,825
ड                              208,515
प                              206,868
म                              206,488
com                            201,464
center                         193,696
ltd                            161,428
स                              160,325
इव                             157,955
partners                       152,694
pvt                            147,583
services                       145,239
group                          139,390
s                              138,303
llc                            121,171
and                            118,201
c                              113,281
india                          110,684
क                              105,891
holdings                       97,019
l                              93,771
care                           88,788
of                 

In [25]:
NAME_STOPWORDS = {
    "inc", "incorporated",
    "llc",
    "ltd", "limited",
    "pvt", "private",
    "company", "co",
    "corp", "corporation",
    "group",
    "services",
    "partners",
    "holdings",
    "associates",
    "center",
    "india",
    "and",
    "of"
}

def meaningful_tokens(name):
    if pd.isna(name) or not str(name).strip():
        return []

    tokens = str(name).split()

    return [
        token for token in tokens
        if token not in NAME_STOPWORDS
        and len(token) >= 2
    ]

In [26]:
examples = [
    "payne enterprises",
    "payne enrtprmises",
    "maure williams colombier",
    "maure wilblims colombier",
    "dahlia power reliable scientific",
    "dahlia power reliable",
    "raj investments",
    "ss food",
    "orellana investments investments"
]

for name in examples:
    print(name, "->", meaningful_tokens(name))

payne enterprises -> ['payne', 'enterprises']
payne enrtprmises -> ['payne', 'enrtprmises']
maure williams colombier -> ['maure', 'williams', 'colombier']
maure wilblims colombier -> ['maure', 'wilblims', 'colombier']
dahlia power reliable scientific -> ['dahlia', 'power', 'reliable', 'scientific']
dahlia power reliable -> ['dahlia', 'power', 'reliable']
raj investments -> ['raj', 'investments']
ss food -> ['ss', 'food']
orellana investments investments -> ['orellana', 'investments', 'investments']


In [27]:
from collections import Counter
import gc
import pandas as pd

meaningful_token_counts = Counter()

for chunk in pd.read_csv(
    S2_PATH,
    sep="\t",
    usecols=["name_core"],
    chunksize=200_000,
    dtype={"name_core": "string"}
):
    for name in chunk["name_core"].dropna():
        for token in set(meaningful_tokens(name)):
            meaningful_token_counts[token] += 1

    del chunk
    gc.collect()

print("Unique meaningful tokens:", len(meaningful_token_counts))

print("\nMost common meaningful tokens:")
for token, count in meaningful_token_counts.most_common(30):
    print(f"{token:30} {count:,}")

Unique meaningful tokens: 810209

Most common meaningful tokens:
com                            201,464
इव                             157,955
care                           88,788
service                        55,261
health                         51,338
enterprises                    49,341
lp                             47,941
industries                     47,928
clinic                         46,246
ventures                       44,338
the                            42,897
solutions                      41,013
public                         36,965
pc                             35,772
global                         35,242
brothers                       35,234
trading                        35,098
pediatric                      32,974
medicine                       31,536
technologies                   30,230
exports                        29,692
यर                             29,249
dental                         26,003
dr                             25,806
shri                 

In [28]:
from collections import Counter

frequency_buckets = {
    "1": 0,
    "2-5": 0,
    "6-10": 0,
    "11-50": 0,
    "51-100": 0,
    "101-500": 0,
    "501-1000": 0,
    "1001-5000": 0,
    "5001-10000": 0,
    "10001-50000": 0,
    "50001+": 0
}

for token, count in meaningful_token_counts.items():

    if count == 1:
        frequency_buckets["1"] += 1
    elif count <= 5:
        frequency_buckets["2-5"] += 1
    elif count <= 10:
        frequency_buckets["6-10"] += 1
    elif count <= 50:
        frequency_buckets["11-50"] += 1
    elif count <= 100:
        frequency_buckets["51-100"] += 1
    elif count <= 500:
        frequency_buckets["101-500"] += 1
    elif count <= 1000:
        frequency_buckets["501-1000"] += 1
    elif count <= 5000:
        frequency_buckets["1001-5000"] += 1
    elif count <= 10000:
        frequency_buckets["5001-10000"] += 1
    elif count <= 50000:
        frequency_buckets["10001-50000"] += 1
    else:
        frequency_buckets["50001+"] += 1

for bucket, count in frequency_buckets.items():
    print(f"{bucket:>12}: {count:,}")

           1: 640,328
         2-5: 88,666
        6-10: 24,300
       11-50: 38,107
      51-100: 9,998
     101-500: 6,224
    501-1000: 1,273
   1001-5000: 911
  5001-10000: 161
 10001-50000: 236
      50001+: 5


In [29]:
from collections import defaultdict
import pandas as pd
import gc

MIN_TOKEN_FREQ = 2
MAX_TOKEN_FREQ = 500

# Keeps only tokens that occur in the useful frequency range
allowed_tokens = {
    token
    for token, count in meaningful_token_counts.items()
    if MIN_TOKEN_FREQ <= count <= MAX_TOKEN_FREQ
}

print("Allowed tokens:", len(allowed_tokens))

Allowed tokens: 167295


In [30]:
def build_token_index(path, allowed_tokens):
    index = defaultdict(list)

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["entity_id", "name_core"],
        chunksize=200_000,
        dtype={
            "entity_id": "string",
            "name_core": "string"
        }
    ):
        for entity_id, name in zip(
            chunk["entity_id"],
            chunk["name_core"]
        ):
            if pd.isna(name):
                continue

            tokens = set(meaningful_tokens(name))

            for token in tokens:
                if token in allowed_tokens:
                    index[token].append(entity_id)

        del chunk
        gc.collect()

    return index

In [31]:
print("Building S2 token index...")
s2_token_index = build_token_index(
    S2_PATH,
    allowed_tokens
)

print("S2 token blocks:", len(s2_token_index))

Building S2 token index...
S2 token blocks: 167295


In [32]:
def calculate_token_block_recall(
    s1_df,
    gt_lookup,
    token_index,
    prefix
):
    total_matches = 0
    captured_matches = 0

    for row in s1_df.itertuples(index=False):

        s1_id = row.entity_id
        name = row.name_core

        true_ids = {
            x for x in gt_lookup.get(s1_id, set())
            if x.startswith(prefix)
        }

        if not true_ids:
            continue

        total_matches += len(true_ids)

        if pd.isna(name):
            continue

        tokens = set(meaningful_tokens(name))

        candidate_set = set()

        for token in tokens:
            if token in allowed_tokens:
                candidate_set.update(token_index.get(token, []))

        captured_matches += len(
            true_ids.intersection(candidate_set)
        )

    recall = (
        captured_matches / total_matches
        if total_matches else 0
    )

    return total_matches, captured_matches, recall

In [33]:
s2_token_total, s2_token_captured, s2_token_recall = \
    calculate_token_block_recall(
        s1,
        gt_lookup,
        s2_token_index,
        "S2-")

print("S2 total true matches:", s2_token_total)
print("S2 captured matches:", s2_token_captured)
print(f"S2 token blocking recall: {s2_token_recall:.4%}")

S2 total true matches: 3693619
S2 captured matches: 1568457
S2 token blocking recall: 42.4640%


In [34]:
def token_candidate_stats(s1_df, token_index, sample_size=10000):
    candidate_counts = []

    for i, row in enumerate(s1_df.itertuples(index=False)):

        if i >= sample_size:
            break

        name = row.name_core

        if pd.isna(name):
            candidate_counts.append(0)
            continue

        tokens = set(meaningful_tokens(name))

        candidate_set = set()

        for token in tokens:
            if token in allowed_tokens:
                candidate_set.update(
                    token_index.get(token, [])
                )

        candidate_counts.append(len(candidate_set))

    s = pd.Series(candidate_counts)

    return {
        "sample_size": len(s),
        "mean": s.mean(),
        "median": s.median(),
        "p90": s.quantile(0.90),
        "p95": s.quantile(0.95),
        "p99": s.quantile(0.99),
        "max": s.max(),
        "zero_candidates": (s == 0).sum()
    }

In [35]:
s2_token_stats = token_candidate_stats(
    s1,
    s2_token_index,
    sample_size=10000
)

s2_token_stats

{'sample_size': 10000,
 'mean': np.float64(69.5515),
 'median': 0.0,
 'p90': np.float64(242.10000000000036),
 'p95': np.float64(357.0),
 'p99': np.float64(517.0),
 'max': 995,
 'zero_candidates': np.int64(5161)}

In [36]:
from collections import defaultdict
import pandas as pd
import gc

def build_name_index(path):
    index = defaultdict(list)

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["entity_id", "country_norm", "name_core"],
        chunksize=200_000,
        dtype={
            "entity_id": "string",
            "country_norm": "string",
            "name_core": "string"
        }
    ):
        chunk = chunk[
            chunk["name_core"].notna() &
            (chunk["name_core"] != "")
        ]

        for country, name, entity_id in zip(
            chunk["country_norm"],
            chunk["name_core"],
            chunk["entity_id"]
        ):
            index[(country, name)].append(entity_id)

        del chunk
        gc.collect()

    return index

In [37]:
print("Building S2 exact-name index...")

s2_name_index = build_name_index(S2_PATH)

print("S2 name blocks:", len(s2_name_index))

Building S2 exact-name index...
S2 name blocks: 3597772


In [38]:
def calculate_union_recall(
    s1_df,
    gt_lookup,
    name_index,
    token_index,
    allowed_tokens,
    prefix
):
    total_matches = 0
    captured_matches = 0

    for row in s1_df.itertuples(index=False):
        s1_id = row.entity_id
        country = row.country_norm
        name = row.name_core
        true_ids = {
            x for x in gt_lookup.get(s1_id, set())
            if x.startswith(prefix)
        }

        if not true_ids:
            continue

        total_matches += len(true_ids)
        candidate_set = set()

        if pd.notna(name):
            candidate_set.update(
                name_index.get((country, name), [])
            )

        if pd.notna(name):

            tokens = set(meaningful_tokens(name))
            for token in tokens:
                if token in allowed_tokens:
                    candidate_set.update(
                        token_index.get(token, [])
                    )

        captured_matches += len(
            true_ids.intersection(candidate_set)
        )

    recall = (
        captured_matches / total_matches
        if total_matches else 0
    )

    return total_matches, captured_matches, recall

In [39]:
s2_union_total, s2_union_captured, s2_union_recall = \
    calculate_union_recall(
        s1,
        gt_lookup,
        s2_name_index,
        s2_token_index,
        allowed_tokens,
        "S2-"
    )

print("S2 total true matches:", s2_union_total)
print("S2 captured matches:", s2_union_captured)
print(f"S2 union blocking recall: {s2_union_recall:.4%}")

S2 total true matches: 3693619
S2 captured matches: 2255373
S2 union blocking recall: 61.0613%
